In [20]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

sys.path.append(str(Path().resolve().parent))

from src.data import get_data
from src.features import generate_features
from src.backtest import backtest, results_table, optimize, optimize_assets, fit_strategy, sharpe_ratio, optimize_tt
from src.strategy import (
    logistic_strategy, 
    momentum_strategy, 
    momentum_volatility_strategy, 
    mean_reversion_strategy, 
    momentum_trend_strategy, 
    momentum_voluratio_strategy,
    randforest_strategy
)
from src.visualization import plot_performance
from src.ml import build_ml_dataset, train_logistic, train_randforest
from src.finance import calculate_beta
from src.portfolio import optimize_portfolio, create_portfolio_df, efficient_frontier

In [21]:
assets = ['AAPL', 'MSFT','SPY', 'NVDA','TTD','DOCS','IBM','BE','XOM','GLD','JNJ','SCCO','ECL','JPM','META', 'QQQ']

In [22]:
asset_summary = pd.read_csv("../outputs/asset_summary.csv")
asset_summary.sort_values('Best Strategy')

,Asset,Beta,Avg Returns,Avg Volatility,Best Strategy,Best Test Sharpe
1,MSFT,1.178747,0.001009,0.015196,Logistic Regression,1.218092
2,NVDA,1.798033,0.002657,0.028976,Logistic Regression,1.972225
10,SCCO,1.102965,0.000959,0.021429,Logistic Regression,0.911062
13,META,1.307990,0.001040,0.021412,Logistic Regression,1.884032
3,TTD,1.773262,0.002009,0.038760,Momentum,0.448013
5,IBM,0.821961,0.000451,0.013995,Momentum,1.778768
9,JNJ,0.462261,0.000355,0.010314,Momentum,1.396559
4,DOCS,1.488148,0.000750,0.038755,Momentum + Rolling Volatility,1.024407
12,JPM,1.086492,0.000797,0.014996,Momentum + Rolling Volatility,1.916430
6,BE,1.842010,0.002372,0.053618,Momentum + Trend Strength,1.442981


In [23]:
grouped = asset_summary.groupby("Best Strategy")["Asset"].apply(list)

for strategy, asset in grouped.items():
    print(f"{strategy}: {', '.join(asset)}")


Logistic Regression: MSFT, NVDA, SCCO, META
Momentum: TTD, IBM, JNJ
Momentum + Rolling Volatility: DOCS, JPM
Momentum + Trend Strength: BE
Random Forest: AAPL, XOM, GLD, ECL


## Strategy Characteristic Study

In [24]:
df = pd.read_csv("../outputs/experiment_results.csv").copy()
df['Generalization Ratio'] = (df['Test Sharpe'] - df['Train Sharpe']) / df['Train Sharpe']
df.head()

,Asset,Strategy,Feature Set,Window,Threshold,Train Sharpe,Train Volatility,Train Max DD,Train Signal Accuracy,Train Signal Precision,Train Signal Recall,Train Signal Active %,Test Sharpe,Test Volatility,Test Max DD,Test Signal Accuracy,Test Signal Precision,Test Signal Recall,Test Signal Active %,Generalization Ratio
0,AAPL,Momentum,NaN,14.0,NaN,1.416952,0.199519,-0.313157,0.525867,0.543907,0.651288,63.445139,1.212668,0.174526,-0.169209,0.532450,0.565310,0.637681,61.854305,-0.144172
1,AAPL,Mean Reversion,NaN,4.0,2.0,1.028301,0.136398,-0.118219,0.484935,0.585526,0.095494,8.641273,-0.209006,0.118274,-0.235419,0.450331,0.489796,0.057971,6.490066,-1.203254
2,AAPL,Momentum + Trend Strength,NaN,10.0,NaN,1.197057,0.151061,-0.144578,0.509380,0.545575,0.443133,43.035816,1.026926,0.120937,-0.123670,0.490066,0.546032,0.415459,41.721854,-0.142124
3,AAPL,Momentum + Rolling Volatility,NaN,14.0,2.0,1.686844,0.157121,-0.118903,0.522456,0.552154,0.522532,50.142126,1.001642,0.163049,-0.170032,0.509934,0.551643,0.567633,56.423841,-0.406204
4,AAPL,Momentum + Volume Ratio,NaN,20.0,NaN,1.230180,0.135403,-0.141400,0.495736,0.555283,0.242489,23.138147,0.782793,0.118371,-0.145055,0.470199,0.541667,0.219807,22.251656,-0.363676


In [25]:
summary = (
    df.groupby("Strategy")
      .agg({
        "Train Sharpe":"mean",
        "Test Sharpe":"mean",
        "Train Volatility": "mean",
        "Test Volatility":"mean",
        "Generalization Ratio":"mean"
      })
      .sort_values(
          "Test Sharpe",
          ascending=False
      )
)

summary

,Train Sharpe,Test Sharpe,Train Volatility,Test Volatility,Generalization Ratio
Strategy,,,,,
Momentum,0.710472,0.969023,0.259419,0.268050,0.794371
Momentum + Rolling Volatility,0.820345,0.932915,0.231243,0.262417,0.438721
Logistic Regression,0.633086,0.900300,0.291169,0.191574,1.145068
Random Forest,8.645609,0.827776,0.248711,0.225066,-0.906754
Momentum + Trend Strength,0.519231,0.677825,0.207372,0.227665,-0.907412
Momentum + Volume Ratio,0.466498,0.498473,0.181039,0.191037,0.445975
Mean Reversion,0.809215,-0.037417,0.139963,0.070043,-1.100116


In [26]:
display(df[df['Strategy'] == 'Momentum'].sort_values('Test Sharpe', ascending = False))

,Asset,Strategy,Feature Set,Window,Threshold,Train Sharpe,Train Volatility,Train Max DD,Train Signal Accuracy,Train Signal Precision,Train Signal Recall,Train Signal Active %,Test Sharpe,Test Volatility,Test Max DD,Test Signal Accuracy,Test Signal Precision,Test Signal Recall,Test Signal Active %,Generalization Ratio
42,IBM,Momentum,NaN,12.0,NaN,0.426745,0.171460,-0.267153,0.509949,0.528796,0.550709,54.292211,1.778768,0.178565,-0.123806,0.547020,0.595238,0.592417,55.629139,3.168218
14,SPY,Momentum,NaN,12.0,NaN,0.949111,0.107847,-0.137841,0.526435,0.553173,0.673981,66.287663,1.557445,0.092455,-0.068494,0.539073,0.579655,0.700696,69.006623,0.640952
63,GLD,Momentum,NaN,4.0,NaN,0.644101,0.099748,-0.206366,0.508243,0.536431,0.543897,53.837408,1.472177,0.132889,-0.110646,0.507285,0.556818,0.580569,58.278146,1.285629
21,NVDA,Momentum,NaN,12.0,NaN,1.177565,0.338559,-0.397698,0.513928,0.544304,0.631689,62.876634,1.472070,0.371944,-0.405710,0.509934,0.549383,0.638756,64.370861,0.250096
70,JNJ,Momentum,NaN,16.0,NaN,0.322750,0.119611,-0.181099,0.488346,0.502964,0.561810,57.532689,1.396559,0.118164,-0.068093,0.541722,0.565217,0.527919,48.741722,3.327057
0,AAPL,Momentum,NaN,14.0,NaN,1.416952,0.199519,-0.313157,0.525867,0.543907,0.651288,63.445139,1.212668,0.174526,-0.169209,0.532450,0.565310,0.637681,61.854305,-0.144172
49,BE,Momentum,NaN,8.0,NaN,0.982980,0.620901,-0.479814,0.515661,0.488889,0.496774,48.128342,1.209882,0.764755,-0.504232,0.506239,0.516234,0.554007,54.901961,0.230831
98,META,Momentum,NaN,48.0,NaN,0.490105,0.224835,-0.268660,0.511086,0.526616,0.604803,59.806708,1.011571,0.300083,-0.267216,0.498013,0.520073,0.710723,72.582781,1.063988
7,MSFT,Momentum,NaN,36.0,NaN,0.674419,0.189500,-0.235351,0.523024,0.540284,0.726886,71.972712,0.949769,0.175276,-0.174708,0.488742,0.523504,0.600490,61.986755,0.408278
91,JPM,Momentum,NaN,46.0,NaN,0.721513,0.175182,-0.261025,0.504832,0.505116,0.615646,61.114269,0.920057,0.166483,-0.171771,0.536424,0.572222,0.721963,71.523179,0.275177


In [27]:
display(df[df['Strategy'] == 'Mean Reversion'].sort_values('Test Sharpe', ascending = False))

,Asset,Strategy,Feature Set,Window,Threshold,Train Sharpe,Train Volatility,Train Max DD,Train Signal Accuracy,Train Signal Precision,Train Signal Recall,Train Signal Active %,Test Sharpe,Test Volatility,Test Max DD,Test Signal Accuracy,Test Signal Precision,Test Signal Recall,Test Signal Active %,Generalization Ratio
22,NVDA,Mean Reversion,NaN,6.0,8.0,0.962589,0.120797,-0.119990,0.466174,0.694444,0.026233,2.046617,1.277397,0.105023,-0.016574,0.456954,0.900000,0.021531,1.324503,0.327044
8,MSFT,Mean Reversion,NaN,4.0,2.0,1.174857,0.127896,-0.111437,0.476407,0.579365,0.077577,7.163161,0.389508,0.065414,-0.059274,0.460927,0.512821,0.049020,5.165563,-0.668464
99,META,Mean Reversion,NaN,14.0,2.0,0.832727,0.232620,-0.272569,0.512791,0.574684,0.247817,22.455941,0.333164,0.200296,-0.217802,0.478146,0.523490,0.194514,19.735099,-0.599912
85,ECL,Mean Reversion,NaN,20.0,8.0,0.532271,0.129678,-0.261239,0.477544,0.594595,0.023758,2.103468,0.331466,0.032665,-0.020319,0.455629,0.500000,0.002433,0.264901,-0.377261
36,DOCS,Mean Reversion,NaN,14.0,6.0,1.015505,0.413007,-0.361454,0.535849,0.560000,0.217054,18.867925,0.061354,0.191252,-0.144962,0.445748,0.450980,0.125000,14.956012,-0.939582
29,TTD,Mean Reversion,NaN,24.0,8.0,0.958389,0.346475,-0.429153,0.489896,0.564854,0.156250,14.635640,-0.075601,0.225366,-0.345825,0.481429,0.545455,0.176471,17.285714,-1.078883
1,AAPL,Mean Reversion,NaN,4.0,2.0,1.028301,0.136398,-0.118219,0.484935,0.585526,0.095494,8.641273,-0.209006,0.118274,-0.235419,0.450331,0.489796,0.057971,6.490066,-1.203254
78,SCCO,Mean Reversion,NaN,4.0,6.0,0.839036,0.078957,-0.005604,0.488346,0.750000,0.006637,0.454804,-0.410612,0.028440,-0.047481,0.468874,0.500000,0.004988,0.529801,-1.489385
15,SPY,Mean Reversion,NaN,4.0,4.0,0.707663,0.056532,-0.025568,0.457646,0.636364,0.007315,0.625355,-0.639831,0.009106,-0.017416,0.426490,0.000000,0.000000,0.264901,-1.904147
50,BE,Mean Reversion,NaN,2.0,14.0,0.544808,0.189085,0.000000,0.528648,1.000000,0.004839,0.229183,-0.670820,0.060386,-0.090098,0.486631,0.000000,0.000000,0.178253,-2.231297
